# Traitement et stockage des données issues du scraping

Vous avez scrapé les données du site de livres et les avez stockées dans un fichier. 

L'objectif de ce notebook est de créer une base de données pour y stocker ces données.

In [40]:
import sqlite3
import pandas as pd

Lire les données du fichier sauvegardé en utilisant pandas.

In [41]:
# Lire les données du fichier que vous venez d'enregistrer
df_books = pd.read_csv("books_info.csv")

## 1. Prétraitement des données

On souhaite créer la table _book_ contenant les attributs suivants : 
- id : INT, PK,
- title : TEXT,
- price : DECIMAL
- availability : BOOLEAN
- rating : INT [0:5]

Vérifier les types des colonnes du dataframe.

In [42]:
# Vérification des types de données
print(df_books.dtypes)

title           object
price           object
rating          object
availability    object
dtype: object


Dans les cellules qui suivent, des méthodes de traitement de données sont suggérées pour donner un aperçu de ce qu'il est possible de faire avec pandas.

**Il est tout à fait possible de faire autrement.**

Utiliser la méthode pandas [_astype_](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.astype.html) pour convertir la colonne de titre en chaîne de caractère.

In [43]:
# Conversion de title en chaîne de caractères
df_books["title"] = df_books["title"].astype(str)

# Vérification du type de la colonne title
print(df_books["title"].dtype)
print(df_books["rating"].head())

object
0    Three
1      One
2      One
3     Four
4     Five
Name: rating, dtype: object


Pour convertir la colonne de prix en nombre décimal, il est nécessaire d'utiliser une étape intermédiaire pour retirer le caractère "£".

Il est possible par exemple d'utiliser l'attribut [.str](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.str.html) de la série "price".

In [44]:
# Convertir la colonne price en type décimal
df_books["price"] = df_books["price"].str.replace('£', '').astype(float)

# Vérification du type de la colonne price
print(df_books["price"].head())
print(df_books["price"].astype)

0    51.77
1    53.74
2    50.10
3    47.82
4    54.23
Name: price, dtype: float64
<bound method NDFrame.astype of 0      51.77
1      53.74
2      50.10
3      47.82
4      54.23
       ...  
995    55.53
996    57.06
997    16.97
998    53.98
999    26.08
Name: price, Length: 1000, dtype: float64>


Convertir la colonne `availability` en boolen (True/False).

Quelles sont les valeurs possibles pour la colonne availability ?

In [45]:
# Valeurs possibles de la colonne availability
print(df_books["availability"].head())

# Afficher les valeurs uniques
print(df_books['availability'].unique())



0    In stock
1    In stock
2    In stock
3    In stock
4    In stock
Name: availability, dtype: object
['In stock']


Créer une fonction qui prend en entrée la valeur de `availability` et qui renvoie True ou False en fonction de la valeur d'entrée.

In [46]:
# Fonction pour convertir la valeur de availability en booléen
def convert_availability(value : str) -> bool:
    """Convert the availability value to a boolean.

    Args:
        value (str): The availability status of the book.

    Returns:
        bool: True if the book is available, False otherwise.
    """
    
    if isinstance(value , str) and "in stock" in value.lower().strip():
        return True
    else:
        return False

Utiliser la méthode [`apply`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.apply.html) pour appliquer la fonction à la colonne `availability`.

In [47]:
# Convertir la colonne availability en booléen (True/False)
df_books["availability"] = df_books["availability"].apply(convert_availability)

# Vérification du type de la colonne availability
print(df_books["availability"].dtype)
print(df_books["availability"].head())
print(df_books["availability"].unique())

bool
0    True
1    True
2    True
3    True
4    True
Name: availability, dtype: bool
[ True]


Convertir la colonne _rating_ en chiffre en utilisant un dictionnaire `rating_map` et la méthode [_map_](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.map.html).

In [48]:
# Dictionnaire associant les notes au format initial et les valeurs numérique
ratings_map = {
    'Zero': 0,
    'One': 1,
    'Two': 2,
    'Three': 3,
    'Four': 4,
    'Five': 5
}

df_books["rating"] = df_books["rating"].map(ratings_map)
print(df_books["rating"].unique())

# Vérification du type de la colonne rating
# print(df_books["rating"].head(20))

[3 1 4 5 2]


In [49]:
# Créer une fonction convert_types qui combine les traitements faits dans les cellules précédentes
def convert_types(df_books: pd.DataFrame) -> pd.DataFrame:
    """Convert the types of the DataFrame columns to appropriate types.

    Args:
        df_books (pd.DataFrame): The DataFrame containing book data.

    Returns:
        pd.DataFrame: The DataFrame with converted types.
    """
    # Vérification des types de données
    print(df_books.dtypes)

    # Conversion de title en chaîne de caractères
    df_books["title"] = df_books["title"].astype(str)
    # # Convertir la colonne price en type décimal
    df_books["price"] = df_books["price"].str.replace('£', '').astype(float)
    print(df_books["price"].head())

    # # Valeurs possibles de la colonne availability
    # # Afficher les valeurs uniques
    print(df_books['availability'].unique())

    # # Convertir la colonne availability en booléen (True/False)
    df_books["availability"] = df_books["availability"].apply(convert_availability)
    print(df_books['availability'].head())
        
    # Affiche les valeurs uniques originales avant mapping
    print(df_books["rating"].unique())

    # # Convertir la colonne _rating_ en chiffre en utilisant un dictionnaire `rating_map` et la méthode [_map_]
    # Dictionnaire associant les notes au format initial et les valeurs numérique
    ratings_map = {
        'Zero': 0,
        'One': 1,
        'Two': 2,
        'Three': 3,
        'Four': 4,
        'Five': 5
    }
    # df_books["rating"] = df_books["rating"].astype(str).str.strip().str.capitalize()

    df_books["rating"] = df_books["rating"].map(ratings_map)
    print(df_books["rating"].value_counts(dropna=False))


In [50]:
df_books = pd.read_csv("books_info.csv")
convert_types(df_books)

title           object
price           object
rating          object
availability    object
dtype: object
0    51.77
1    53.74
2    50.10
3    47.82
4    54.23
Name: price, dtype: float64
['In stock']
0    True
1    True
2    True
3    True
4    True
Name: availability, dtype: bool
['Three' 'One' 'Four' 'Five' 'Two']
rating
1    226
3    203
5    196
2    196
4    179
Name: count, dtype: int64


---
## 2. Insertion des données en base

Dans cette section :
- on créé une BDD sqlite  `book_store.db` (ou on se connecte à la base si elle existe déjà) en utilisant la bibliothèque python sqlite3,
- on insère les données prétraitées dans la BDD

Utiliser le [tutoriel](https://www.ionos.fr/digitalguide/sites-internet/developpement-web/sqlite3-avec-python/) pour l'utilisation de sqlite3.

Utiliser la fonction pandas adaptée qui permet d'insérer un dataframe dans une BDD.

In [51]:
# Création de la BDD et insertion des données
connection = sqlite3.connect("book_store.db")
# vérifier la bonne création de la base de données
print(connection.total_changes)
# structurer votre base de données, Cursor permet alors d’envoyer des commandes SQL à votre base de données.
cursor = connection.cursor()
# df_books = pd.read_csv("books_info.csv")
df_books.to_sql(
    name= 'book',
    con=connection,
    if_exists='replace',
    index=True,
    index_label='book_id'
)




0


1000

Vérifier le nombre de livres présents dans la BDD en utilisant sqlite3 et la requête SQL adaptée.

In [52]:
# Compter le nombre de livre dans la BDD
cursor.execute("SELECT COUNT(book_id) FROM book")
resultat = cursor.fetchone()
print(f"Le nombre de livres : {resultat[0]}")


connection.commit()
connection.close()


Le nombre de livres : 1000
